# AI应用基础 · 工单M2.2《轴承寿命预测系统》 — 教师参考版
第7周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m2rul"], check=True)
print("数据已生成: data/")

工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础
M2.2 -> data/bearing_rul.npz (5400, 20)
完成。数据输出目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础/data
数据已生成: data/


## 参考实现 `m22`（含依赖导入）

In [2]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m22():
    import torch
    import torch.nn as nn

    d = np.load("data/bearing_rul.npz")
    X = torch.tensor(d["X"][:, :, None])
    y = torch.tensor(d["y"])
    n = int(len(X) * 0.8)
    Xtr, ytr, Xte, yte = X[:n], y[:n], X[n:], y[n:]

    class RULNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm = nn.LSTM(1, 32, batch_first=True)
            self.fc = nn.Linear(32, 1)

        def forward(self, x):
            return self.fc(self.lstm(x)[0][:, -1, :]).squeeze(-1)

    torch.manual_seed(42)
    model = RULNet()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(60):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(Xtr), ytr)
        loss.backward()
        opt.step()
        if (ep + 1) % 10 == 0:
            print(f"epoch{ep + 1} loss={loss.item():.4f}")
    with torch.no_grad():
        p = model(Xte).numpy()
    mse = float(np.mean((p - yte.numpy()) ** 2))
    print(f"MSE={mse:.4f} RMSE={mse ** 0.5:.4f} "
          f"MAE={float(np.mean(np.abs(p - yte.numpy()))):.4f}")
    print("业务解读: RMSE≈0.1即寿命预测误差约10%退化区间, 可提前1-2个维护"
          "周期介入, 满足预测性维护排程需求。")


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `1` | TODO-1: 全批量(或batch=64)训练, 每10轮打印loss |
|  | （在此处编写你的代码——依据上方注释/文档字符串） |
| `2` | TODO-2: 计算测试集 MSE/RMSE/MAE 三项指标 TODO-3: 选1台设备的退化段, 绘制"真实RUL |
|  | vs 预测RUL"曲线 TODO-4: (对照)线性回归基线: 用窗口均值单特征拟合, 对比MSE TODO-5: |
|  | (业务解读)≤80字: RMSE=0.1对维护计划意味着什么 |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 执行与验证

In [3]:
m22()

epoch10 loss=0.2442


epoch20 loss=0.1573


epoch30 loss=0.1260


epoch40 loss=0.1098


epoch50 loss=0.0925


epoch60 loss=0.0751
MSE=0.0731 RMSE=0.2703 MAE=0.2340
业务解读: RMSE≈0.1即寿命预测误差约10%退化区间, 可提前1-2个维护周期介入, 满足预测性维护排程需求。
